In [1]:
import pandas as pd
import json
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

In [2]:
train_df = pd.read_csv("./processed_data_for_training/train.csv")
f = open("./processed_data_for_training/train_original.json")
train_original_feature_list = json.load(f)

In [3]:
match_col_list = ["xing", "ming", "zihao", "diqu", "jigou_1", "jigou_2", "guanzhi_1", "ren_xian", "ren_sheng", "chushen_1"]

match_col_stroke_list = [f'{col}_stroke' for col in match_col_list]
match_col_pinyin_list = [f'{col}_pinyin' for col in match_col_list]
model_feature_names = match_col_stroke_list + match_col_pinyin_list + ['ming_cnt_diff', 'ming_sim1', 'ming_sim2', "same_year"]

In [5]:
train_feature_list = train_df.to_numpy()

In [6]:
labelled_idx_list = []

In [7]:
X_total = pd.DataFrame(train_feature_list, columns=model_feature_names)

In [8]:
import os
data_dir = './active_learning_results/'
os.makedirs(data_dir, exist_ok=True)

In [12]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

class CommitteeModel:
    def __init__(self, models):
        self.committee_models = models

    def predict_proba(self, X):
        votes = [m.predict(X) for m in self.committee_models]
        votes_mean = np.array(votes).mean(axis=0)
        return np.column_stack([1 - votes_mean, votes_mean])

    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= 0.5).astype(int)

    def member_probas(self, X):
        return np.column_stack([m.predict_proba(X)[:, 1] for m in self.committee_models])


def original_feature_list_to_readable_df(feature_list):
    feature_list_ = []
    for feature_dict in feature_list:
        feature_dict_ = {}
        feature_dict_['name'] = f"{feature_dict['1_xing']}{feature_dict['1_ming']} |\n {feature_dict['2_xing']}{feature_dict['2_ming']}"
        feature_dict_['zihao'] = f"{feature_dict['1_zihao']} |\n {feature_dict['2_zihao']}"
        feature_dict_['diqu'] = f"{feature_dict['1_diqu']} |\n {feature_dict['2_diqu']}"
        feature_dict_['jigou'] = f"{feature_dict['1_jigou_1']} {feature_dict['1_jigou_2']} |\n {feature_dict['2_jigou_1']} {feature_dict['2_jigou_2']}"
        feature_dict_['guanzhi'] = f"{feature_dict['1_guanzhi_1']} |\n {feature_dict['2_guanzhi_1']}"
        # feature_dict_['pinji'] = f"{feature_dict['1_pinji']} |\n {feature_dict['2_pinji']}"
        feature_dict_['chushen'] = f"{feature_dict['1_chushen_1']} |\n {feature_dict['2_chushen_1']}"
        feature_dict_['rensheng&xian'] = f"{feature_dict['1_ren_xian']} {feature_dict['1_ren_sheng']} |\n {feature_dict['2_ren_xian']} {feature_dict['2_ren_sheng']}"
        feature_dict_['oracle'] = None
        feature_dict_['year'] = f"{feature_dict['1_year']} |\n {feature_dict['2_year']}"
        feature_list_.append(feature_dict_)
    return pd.DataFrame(feature_list_)

def get_marginal_df(model, X, not_include_index_list, top_k_uncertainty=5000, top_k_final=100, n_clusters=None):
    """
    First select uncertainty pool by voting entropy, then diversify within the pool (at most 1 per cluster + k-center completion).
    n_clusters defaults to match top_k_final to avoid duplicate samples from same cluster.
    """
    proba_mean = model.predict_proba(X)  # [n,2]
    pred = model.predict(X)

    eps = 1e-12
    if hasattr(model, 'member_probas'):
        P_members = model.member_probas(X)  # [n,B]
        votes = (P_members >= 0.5).astype(int)
        f_pos = votes.mean(axis=1)
        uncertainty = -(f_pos*np.log(f_pos+eps) + (1-f_pos)*np.log(1-f_pos+eps))
    else:
        p = proba_mean[:, 1]
        uncertainty = -(p*np.log(p+eps) + (1-p)*np.log(1-p+eps))

    prob_df = pd.DataFrame({
        0: proba_mean[:, 0],
        1: proba_mean[:, 1],
        'index': X.index,
        'predicted': f_pos,
        'uncertainty': uncertainty
    })

    prob_df = prob_df[~prob_df['index'].isin(not_include_index_list)]

    prob_df = prob_df.sort_values('uncertainty', ascending=False).reset_index(drop=True)
    top_uncertain = prob_df.iloc[:min(top_k_uncertainty, len(prob_df))].copy()
    uncertain_indices = top_uncertain['index'].values

    pool_X = X.loc[uncertain_indices]
    n_clusters_eff = n_clusters if n_clusters is not None else top_k_final
    diverse_local_idx = get_diverse_samples(pool_X, n_clusters=n_clusters_eff, top_k=top_k_final)

    selected_df = top_uncertain.iloc[diverse_local_idx].reset_index(drop=True)
    return selected_df

def get_diverse_samples(X, n_clusters, top_k):
    """
    - Standardization + KMeans clustering, n_clusters clipped to <=min(top_k, n);
    - At most 1 sample per cluster (closest to cluster center, most representative);
    - If not enough samples selected to reach top_k, use k-center greedy algorithm on remaining samples to maximize minimum distance to already selected samples.
    Returns local indices (0..len(X)-1).
    """
    n = len(X)
    if n == 0:
        return []
    if top_k >= n:
        return list(range(n))

    n_clusters_eff = int(max(1, min(n_clusters, top_k, n)))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X.values)

    kmeans = KMeans(n_clusters=n_clusters_eff, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(X_scaled)

    chosen = []
    for c in range(n_clusters_eff):
        idx_c = np.where(cluster_labels == c)[0]
        if len(idx_c) == 0:
            continue
        center = kmeans.cluster_centers_[c]
        dists = np.linalg.norm(X_scaled[idx_c] - center, axis=1)
        chosen.append(idx_c[np.argmin(dists)])

    chosen = list(dict.fromkeys(chosen))
    if len(chosen) >= top_k:
        return chosen[:top_k]


    remaining = np.array([i for i in range(n) if i not in set(chosen)])
    Xs = X_scaled

    if len(chosen) == 0:
        mean_vec = Xs.mean(axis=0)
        start = int(np.argmax(np.linalg.norm(Xs - mean_vec, axis=1)))
        chosen.append(start)
        remaining = remaining[remaining != start]

    def min_dist_to_chosen(cands):
        if len(chosen) == 0:
            return np.full(len(cands), np.inf)
        dmin = np.full(len(cands), np.inf)
        for j, i in enumerate(cands):
            dmin[j] = np.min([np.linalg.norm(Xs[i] - Xs[k]) for k in chosen])
        return dmin

    while len(chosen) < top_k and len(remaining) > 0:
        dmin = min_dist_to_chosen(remaining)
        j = int(np.argmax(dmin))
        next_idx = int(remaining[j])
        chosen.append(next_idx)
        remaining = np.delete(remaining, j)

    return chosen[:top_k]

def train_and_label_round(X_train, y_train, round_num, labelled_idx_list, X_total, model_feature_names, train_original_feature_list, committee_size=25):
    rng = np.random.default_rng(42 + round_num)
    models = []
    n = len(X_train)

    pos = int((np.array(y_train) == 1).sum())
    neg = n - pos
    spw = float(neg / max(pos, 1))

    for b in range(committee_size):
        idx = rng.choice(n, size=n, replace=True)
        m = XGBClassifier(
            objective='binary:logistic',
            missing=-1,
            n_estimators=180,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.7,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            tree_method='hist',
            eval_metric='logloss',
            random_state=int(rng.integers(0, 1 << 31)),
            scale_pos_weight=spw
        )
        m.fit(X_train.iloc[idx], np.array(y_train)[idx])
        models.append(m)

    bst = CommitteeModel(models)

    marginal_idx_list = get_marginal_df(bst, X_total, labelled_idx_list)

    to_label_df = original_feature_list_to_readable_df(np.array(train_original_feature_list)[marginal_idx_list['index']])
    to_label_df['index'] = list(marginal_idx_list['index'])
    to_label_df['predicted'] = list(marginal_idx_list['predicted'])
    to_label_df['uncertainty'] = list(marginal_idx_list['uncertainty'])
    to_label_df.to_csv(f"{data_dir}to_label_df{round_num}.csv", index=False)

    return bst, marginal_idx_list

def update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, round_num):
    label_df = pd.read_csv(f"{data_dir}label_df{round_num}.csv")

    labelled_idx_list = np.concatenate([labelled_idx_list, marginal_idx_list['index']], axis=0)

    X_train = pd.concat([X_train, X_total.iloc[marginal_idx_list['index']]])
    y_train = np.concatenate([y_train, label_df['oracle']], axis=0)

    return X_train, y_train, labelled_idx_list

In [16]:
X_train = X_total.iloc[0:0].copy()
y_train = np.array([])

In [17]:
INIT_ROUND_NUM = 100

np.random.seed(42)

init_train_idx_list = np.random.choice(len(train_original_feature_list), size=INIT_ROUND_NUM, replace=False)
init_to_label_df = original_feature_list_to_readable_df(np.array(train_original_feature_list)[init_train_idx_list])

init_to_label_df['index'] = list(init_train_idx_list)
init_to_label_df.to_csv(f"{data_dir}to_label_df0.csv", index=False)

In [19]:
X_train, y_train, labelled_idx_list = update_training_data(init_to_label_df, labelled_idx_list, X_train, y_train, X_total, 0)

In [20]:
bst1, marginal_idx_list = train_and_label_round(X_train, y_train, 1, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [21]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 1)

In [22]:
bst2, marginal_idx_list = train_and_label_round(X_train, y_train, 2, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [23]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 2)

In [24]:
bst3, marginal_idx_list = train_and_label_round(X_train, y_train, 3, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [26]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 3)

In [27]:
bst4, marginal_idx_list = train_and_label_round(X_train, y_train, 4, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [28]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 4)

In [29]:
bst5, marginal_idx_list = train_and_label_round(X_train, y_train, 5, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [30]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 5)

In [31]:
bst6, marginal_idx_list = train_and_label_round(X_train, y_train, 6, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [33]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 6)

In [34]:
bst7, marginal_idx_list = train_and_label_round(X_train, y_train, 7, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [123]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 7)

In [124]:
bst8, marginal_idx_list = train_and_label_round(X_train, y_train, 8, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [225]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 8)

In [238]:
bst9, marginal_idx_list = train_and_label_round(X_train, y_train, 9, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [324]:
X_train, y_train, labelled_idx_list = update_training_data(marginal_idx_list, labelled_idx_list, X_train, y_train, X_total, 9)

In [325]:
bst10, marginal_idx_list = train_and_label_round(X_train, y_train, 10, labelled_idx_list, X_total, model_feature_names, train_original_feature_list)

In [35]:
rng = np.random.default_rng(42)
models = []
n = len(X_train)

pos = int((np.array(y_train) == 1).sum())
neg = n - pos
spw = float(neg / max(pos, 1))

model_dir = f'{data_dir}ensemble/'
os.makedirs(model_dir, exist_ok=True)

for b in range(25):
    idx = rng.choice(n, size=n, replace=True)
    m = XGBClassifier(
        objective='binary:logistic',
        missing=-1,
        n_estimators=180, subsample=0.7, min_child_weight=2, gamma=0.2, max_depth=4, learning_rate=0.1,
        random_state=int(rng.integers(0, 1 << 31)),
        scale_pos_weight=spw
    )
    m.fit(X_train.iloc[idx], np.array(y_train)[idx])
    m.save_model(f"{model_dir}xgboost_iter7_{b}.json")